# Direct RGB-D/language VLA — GPU training
Attach `direct_vla_source.zip` and `direct_vla_v1.h5` datasets. Enable GPU. This trains a compact two-task recurrent VLA, not an open-world foundation model. CPU smoke checkpoint is not the trained candidate. Save Version with outputs after training.


In [ ]:
from pathlib import Path
import zipfile, subprocess, sys
archives=list(Path('/kaggle/input').rglob('direct_vla_source.zip'))
assert len(archives)==1, archives
with zipfile.ZipFile(archives[0]) as archive: archive.extractall('/kaggle/working/source')
root=Path('/kaggle/working/source/omni_vla')
subprocess.run([sys.executable,'-m','pip','install','-e',str(root)+'[train,deploy,test]'],check=True)


In [ ]:
import torch
print('Torch',torch.__version__,'GPU',torch.cuda.get_device_name() if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), 'Enable GPU for the full training run'
datasets=[p for p in Path('/kaggle/input').rglob('*.h5') if p.name.startswith('direct_vla_')]
assert len(datasets)==1, datasets
dataset=datasets[0]


In [ ]:
subprocess.run([sys.executable,'-m','scripts.train_vla','--dataset',str(dataset),
                '--output','/kaggle/working/direct_vla','--epochs','100',
                '--batch-size','32','--history','8','--device','cuda'],cwd=root,check=True)


In [ ]:
import shutil
shutil.make_archive('/kaggle/working/direct_vla_outputs','zip','/kaggle/working/direct_vla')
print('Download direct_vla_outputs.zip; retain dataset/source hashes in the checkpoint.')


## CPU evaluation after downloading
Extract outputs into a new `weights/direct_vla_kaggle/` directory. Evaluate both tasks and each IID/OOD profile on new seeds. A low validation loss alone is not a success gate. Run image/text/history ablations and multiple training seeds before claiming generalization.

```bash
MUJOCO_GL=egl .venv/bin/python -m scripts.eval_direct_vla --checkpoint weights/direct_vla_kaggle/best.pt --task 0 --episodes 50 --seed 90000 --output artifacts/vla_stow_fresh.json
MUJOCO_GL=egl .venv/bin/python -m scripts.eval_direct_vla --checkpoint weights/direct_vla_kaggle/best.pt --task 1 --episodes 50 --seed 91000 --output artifacts/vla_open_fresh.json
```
